In [1]:
"""
Objective: Use model evaluation to improve the initial
draft of an prompt
"""
# Identify changes in source files and reload
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
from anthropic import Anthropic
import json
from prompt_evaluations_exercise import PromptEvaluations
from evaluation_dataset import EvaluationData
from syntax_validation import SyntaxValidation

In [2]:
# Access the API key
load_dotenv()

True

In [3]:
# Access the Anthropic API
client = Anthropic()
# Specify the model Claude will use
model = "claude-sonnet-5"

In [4]:

"""
Access:
- Conversation history related to structurizing Claude output data
- Functions to store user inputs
- Functions to store Claude responses
"""
promptEvaluations = PromptEvaluations(client, model)
"""
Access:
- Functions to save evaluation datasets
- Functions to load evaluation datasets
"""
evaluationData = EvaluationData('dataset.json')
"""
Access: Helper functions to validate JSON, Python, and regular expression outputs
"""
syntaxValidation = SyntaxValidation()

In [5]:
# Prompt engineering rules to replace prefilling
plain_text_prompt_rules = """
Rules:
- Return only plain text
- Do not use markdown
- Do not include comments
- Do not include explanations
- After the plain text, write END_OF_COMMANDS
"""

json_prompt_rules = """
Rules:
- Return only valid JSON
- Do not use markdown
- Do not include comments
- Do not include explanations outside the JSON
- Do not include trailing commas
- After the JSON, write END_OF_COMMANDS
"""

# Provide Claude with context to customize how Claude responds to user input
evaluation_system_prompt = """
You are an expert regarding AWS and how to create Python function, JSON policy documents,
and regular expressions that interact with AWS. You adhere to industry best practices
"""

promptEvaluations.stop_sequences.append("END_OF_COMMANDS")

## Observe the output of the first prompt draft

In [6]:
# Tasks for the prompt evaluation exercise
task = "Create a Python function to extract the AWS account ID from an ARN"

In [7]:
"""
Starting point for grading and improving prompts
Add the task to the f-string
Add prompt rules after the task to replace prefilling
"""
initial_prompt = f"""
Please provide a solution to the following task:
{task}
{plain_text_prompt_rules}
"""

In [8]:
# Store user prompt in the message history
promptEvaluations.storeUserInputs(initial_prompt)

In [9]:
# Get Claude response
claudeResponse = promptEvaluations.askClaude(evaluation_system_prompt)

In [10]:
# Store Claude response in the message history
promptEvaluations.storeClaudeResponse(claudeResponse)

In [11]:
print(claudeResponse)

import re

def extract_account_id_from_arn(arn):
    pattern = r'^arn:(aws|aws-cn|aws-us-gov):[a-zA-Z0-9\-]+:[a-zA-Z0-9\-]*:(\d{12}):.+$'
    match = re.match(pattern, arn)
    if match:
        return match.group(2)
    return None


## Test the prompt against a dataset

In [12]:
"""
Generate test datasets that contains sample inputs representing the types
of questions or requests the prompt will handle
"""
dataset_prompt = f"""
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate
Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task
that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {{
        "task": "Description of task",
        "format: "JSON",
        "solution_criteria": "Key criteria for evaluating the solution"
    }},
    ...additional
]
```
* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
{plain_text_prompt_rules}
"""

dataset_system_prompt = """
You are an expert in creating accurate, useful, and exhaustive evaludation datasets.
You follow industry best practices when it comes to generating prompt evaluation datasets
"""
    

In [13]:
promptEvaluations.storeUserInputs(dataset_prompt)

In [14]:
claudeGeneratedDataset = promptEvaluations.generateDataset(dataset_system_prompt)

In [15]:
evaluationData.setEvaluationDataset(claudeGeneratedDataset)

In [16]:
print(claudeGeneratedDataset)

[
    {
        "task": "Create a Python function to validate whether a given string is a properly formatted S3 bucket name according to AWS naming rules",
        "format": "Python",
        "solution_criteria": "Function accepts a string and returns True or False; enforces length between 3 and 63 characters; only allows lowercase letters, numbers, dots, and hyphens; must start and end with a letter or number; does not allow consecutive periods; does not allow IP address format"
    },
    {
        "task": "Write a regex to match and extract the region and service from an AWS ARN string",
        "format": "Regex",
        "solution_criteria": "Regex correctly matches strings starting with arn: followed by partition, service, region, account-id, and resource; captures service and region as named or numbered groups; handles optional empty region field"
    },
    {
        "task": "Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket

### Feed Evaluation dataset through Claude

In [17]:
dataset = evaluationData.getEvaluationDataset()

In [18]:
for data in dataset:
        prompt = f"""
        Please provide a solution to the following task:
        {data}
        {plain_text_prompt_rules}
        """
        promptEvaluations.storeUserInputs(prompt)

        claudeResponse = promptEvaluations.askClaude(evaluation_system_prompt)

        promptEvaluations.storeClaudeResponse(claudeResponse)

In [19]:
# Verify that the generated code has valid syntax and follows the correct format
def gradeSyntax(task, generated_output):
    format = task["format"]
    synax_score = 0

    if format == "Python":
        synax_score = syntaxValidation.validate_regex(generated_output)

    elif format == "JSON":
        synax_score = syntaxValidation.validate_json(generated_output)

    else:
        synax_score = syntaxValidation.validate_regex(generated_output)

    return synax_score

## Evaluate Claude's output using Claude

In [20]:
model_grading_results = []

grading_prompt = f"""
Evalulate this AI-generated solution
"""

system_prompt = "You are an expert code reviewer"

In [21]:
dataset = evaluationData.getEvaluationDataset()

In [22]:
for data in dataset:
    solution_criteria = data["solution_criteria"]
    
    prompt = f"""
    Please provide a solution to the following task:
    {data}
    Criteria you should use to evaluation the solution
    {solution_criteria}
    {plain_text_prompt_rules}
    """
    
    promptEvaluations.storeUserInputs(prompt)

    claudeResponse = promptEvaluations.askClaude(evaluation_system_prompt)

    promptEvaluations.storeClaudeResponse(claudeResponse)

    # Used this strict shape to ensure low max_token limits do not cause errors
    model_grading_prompt = f"""
    {grading_prompt}
    Task: {data}
    Solution: {claudeResponse}

    Return only valid JSON in this exact shape:
    {{
    "strengths": ["very short strength"],
    "weaknesses": ["very short weakness"],
    "reasoning": "One very short sentence.",
    "score": 7
    }}

    {json_prompt_rules}
    """

    promptEvaluations.storeUserInputs(model_grading_prompt)

    claudeGradingResult = promptEvaluations.askClaude(system_prompt)

    claudeJSONGradingResult = json.loads(claudeGradingResult)

    syntax_score = gradeSyntax(data, claudeResponse)

    claudeJSONGradingResult["score"] += syntax_score

    model_grading_results.append(promptEvaluations.generateTestCaseReport(data, claudeResponse, claudeJSONGradingResult))

    promptEvaluations.storeClaudeResponse(claudeGradingResult)

    scores = [result["score"] for result in model_grading_results]


In [23]:
average = promptEvaluations.calculateAverage(scores)

In [24]:
print(f"Average score: {average}")

Average score: 19.333333333333332
